# DCA Backtest Lab

Configurable monthly DCA + profit-taking + progressive dip buying.

**Edit the parameters below, then Run All.**

In [1]:
# ===== PARAMETERS =====
# Edit these values before running

PAIRS = [
    # (symbol, monthly_invest, reinvest_start%, reinvest_step%, reinvest_cap%)
    ('BTCUSDT', 20, 1, 2, 50),
    ('BNBUSDT', 10, 1, 1, 40),
]

TP_MULTIPLIER = 1.5       # Sell when price reaches ATH * this (e.g. 1.5x, 2.0)
TP_PERCENTAGE = 0.35      # Fraction of position to sell (e.g. 0.35 = 35%)

# ===== END PARAMETERS =====
monthly_total = sum(p[1] for p in PAIRS)
print('PAIRS:', [(p[0], f'${p[1]}', f'{p[2]}%+{p[3]}%->{p[4]}%') for p in PAIRS])
print(f'TP: {TP_PERCENTAGE*100:.0f}% sell at {TP_MULTIPLIER}x ATH')
print(f'Monthly total: ${monthly_total}')

PAIRS: [('BTCUSDT', '$20', '1%+2%->50%'), ('BNBUSDT', '$10', '1%+1%->40%')]
TP: 35% sell at 1.5x ATH
Monthly total: $30


In [2]:
import pandas as pd, requests, time, numpy as np, matplotlib, warnings
matplotlib.use('Agg')
import matplotlib.pyplot as plt, matplotlib.dates as mdates
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print('Libraries loaded.')

Libraries loaded.


In [3]:
def fetch(s):
    url='https://api.binance.com/api/v3/klines'; ak=[]; st=0
    while True:
        p={'symbol': s, 'interval': '1M', 'startTime': st, 'limit': 500}
        d=requests.get(url, params=p).json()
        if not d: break
        ak.extend(d)
        if len(d)<500: break
        st=d[-1][0]+1; time.sleep(0.1)
    return ak

cols=['ts','o','h','l','c','v','ct','qv','t','tbb','tbq','ig']
data={}
for sym,_,_,_,_ in PAIRS:
    df=pd.DataFrame(fetch(sym), columns=cols)
    df['date']=pd.to_datetime(df['ts'], unit='ms', utc=True)
    for c in ['o','h','l','c','v']: df[c]=df[c].astype(float)
    df=df[['date','o','h','l','c','v']].copy().sort_values('date').reset_index(drop=True)
    data[sym]=df
    print(f'{sym}: {df["date"].min().strftime("%Y-%m")} -> {df["date"].max().strftime("%Y-%m")} ({len(df)} mo)')

BTCUSDT: 2017-08 -> 2026-07 (108 mo)


BNBUSDT: 2017-11 -> 2026-07 (105 mo)


In [4]:
# Align all pairs to common date range
start=data[PAIRS[0][0]]['date'].max() if len(PAIRS)==1 else max(d['date'].min() for d in data.values())
end=min(d['date'].max() for d in data.values())
for sym,_,_,_,_ in PAIRS:
    data[sym]=data[sym][(data[sym]['date']>=start)&(data[sym]['date']<=end)].reset_index(drop=True)
months=len(data[PAIRS[0][0]])
print(f'Common range: {start.strftime("%Y-%m")} -> {end.strftime("%Y-%m")} ({months} mo)')

Common range: 2017-11 -> 2026-07 (105 mo)


In [5]:
# --- BACKTEST ENGINE ---

# State per pair
state={}
for sym,_,rs,_,rc in PAIRS:
    state[sym]={'units': 0.0, 'ath': 0.0, 'ath_sell': 0.0, 'reinvest': float(rs), 'cap': float(rc)}

usdt=0.0; total_inj=0.0; total_dip=0.0; total_base=0.0
rec=[]

for i in range(months):
    usdt+=monthly_total; total_inj+=monthly_total
    act=[]

    for sym,minv,rs,rt,rc in PAIRS:
        s=state[sym]
        row=data[sym].iloc[i]
        c=row['c']; o=row['o']
        step=float(rt); cap=float(rc)

        # 1. Progressive dip buy (from reserve)
        if c<o and usdt>10:
            ext=min(usdt*(s['reinvest']/100.0), usdt-monthly_total)
            if ext>5:
                s['units']+=ext/c; usdt-=ext; total_dip+=ext
                act.append(f'{sym[:3]} dip ${ext:.0f}')

        # 2. Monthly DCA base buy
        if usdt>=minv:
            s['units']+=minv/c; usdt-=minv; total_base+=minv
            act.append(f'{sym[:3]} DCA ${minv} @ {c:.0f}')

        # 3. Update ATH
        if c>s['ath']: s['ath']=c

        # 4. Sell at TP_MULTIPLIER * previous ATH
        if s['units']>0.000001 and c>=s['ath_sell']*TP_MULTIPLIER and s['ath_sell']>0:
            sell_units=s['units']*TP_PERCENTAGE
            usdt+=sell_units*c; s['units']-=sell_units
            s['reinvest']=float(rs)  # reset
            s['ath_sell']=s['ath']
            act.append(f'{sym[:3]} SELL {TP_PERCENTAGE*100:.0f}% @ {c:.0f}')
        elif s['ath_sell']==0:
            s['ath_sell']=c
        else:
            if s['reinvest']<cap:
                s['reinvest']=min(cap, s['reinvest']+step)

    # Record
    pf=sum(state[sym]['units']*data[sym].iloc[i]['c'] for sym,_,_,_,_ in PAIRS)+usdt
    row_data={'date': row['date'], 'pf': pf, 'inj': total_inj, 'usdt': usdt}
    for sym,_,_,_,_ in PAIRS:
        s=state[sym]; c=data[sym].iloc[i]['c']
        row_data[f'{sym[:3]}_units']=s['units']
        row_data[f'{sym[:3]}_val']=s['units']*c
        row_data[f'{sym[:3]}_reinvest']=s['reinvest']
    row_data['act']=' | '.join(act) if act else 'nothing'
    rec.append(row_data)

res=pd.DataFrame(rec)
f=res.iloc[-1]
pf_val=f['pf']; inj=f['inj']
print(f'Backtest complete: {len(res)} months')

Backtest complete: 105 months


In [6]:
# --- METRICS ---
eq=res['pf']
ddr=(eq.cummax()-eq)/eq.cummax(); maxdd=ddr.max()*100
mret=eq.pct_change().dropna()
sh=(mret.mean()/mret.std())*np.sqrt(12) if mret.std()>0 else 0
p=mret[mret>0].sum(); ne=mret[mret<0].abs().sum(); pfr=p/ne if ne>0 else float('inf')
ann=(1+(pf_val/inj-1))**(12/len(res))-1
cal=ann/(maxdd/100) if maxdd>0 else 0
win=mret[mret>0]; lose=mret[mret<0]
wr=len(win)/(len(win)+len(lose))*100 if (len(win)+len(lose))>0 else 0
avg_w=win.mean()*100; avg_l=lose.mean()*100

sells=res[res['act'].str.contains('SELL', na=False)]
dip_mo=res[res['act'].str.contains('dip', na=False)]

print('='*72)
print('  DCA BACKTEST LAB - RESULTS')
print('='*72)
print()
print(f'  PERIOD:      {res["date"].min().strftime("%Y-%m-%d")} to {res["date"].max().strftime("%Y-%m-%d")}')
print(f'  DURATION:    {len(res)} months ({len(res)//12}y {len(res)%12}m)')
print(f'  INJECTION:   ${monthly_total}/mo x {len(res)}mo = ${inj:,.0f} total')
print(f'  TP:          {TP_PERCENTAGE*100:.0f}% sell at {TP_MULTIPLIER}x ATH')
print()
print('  PAIRS:')
for sym,minv,rs,rt,rc in PAIRS:
    s=state[sym]; c=data[sym].iloc[-1]['c']; val=s['units']*c
    cost=total_base*minv/monthly_total if monthly_total>0 else 0
    ret=(val/cost-1)*100 if cost>0 else 0
    print(f'    {sym:10s} ${val:>8,.0f} ({s["units"]:>10.6f} @ ${c:>7,.0f}) {ret:>7.1f}% return')
print(f'    USDT rsv:  ${f["usdt"]:>8,.0f} ({f["usdt"]/pf_val*100:.1f}% of pf)')
print()
print(f'  PORTFOLIO:   ${pf_val:>10,.2f}')
print(f'  P&L:         ${pf_val-inj:>10,.2f}')
print(f'  RETURN:      {(pf_val/inj-1)*100:>8.2f}%   (ann {ann*100:.2f}%)')
print(f'  MAX DD:      {maxdd:>8.2f}%')
print(f'  SHARPE:      {sh:>8.2f}')
print(f'  PROFIT FACT: {pfr:>8.2f}')
print(f'  CALMAR:      {cal:>8.2f}')
print(f'  WIN RATE:    {wr:>7.1f}%  ({len(win)}/{len(mret)} months)')
print(f'  AVG WIN:     {avg_w:>7.2f}%   AVG LOSS: {avg_l:>7.2f}%')
print(f'  BEST MONTH:  {mret.max()*100:>7.2f}%   WORST:    {mret.min()*100:>7.2f}%')
print()
print(f'  SELLS:       {len(sells)} events')
for _, r in sells.iterrows():
    sell_acts=[x for x in r['act'].split(' | ') if 'SELL' in x]
    for sa in sell_acts:
        print(f'    {r["date"].strftime("%Y-%m")}: {sa}')
print()
print(f'  DIP BUYS:    {len(dip_mo)} / {len(res)} months ({len(dip_mo)/len(res)*100:.0f}%)')
if total_dip>0:
    print(f'  Total dipped:${total_dip:>8,.0f} (from sell profits)')

  DCA BACKTEST LAB - RESULTS

  PERIOD:      2017-11-01 to 2026-07-01
  DURATION:    105 months (8y 9m)
  INJECTION:   $30/mo x 105mo = $3,150 total
  TP:          35% sell at 1.5x ATH

  PAIRS:
    BTCUSDT    $  13,680 (  0.214339 @ $ 63,826)   551.4% return
    BNBUSDT    $   7,961 ( 13.738853 @ $    579)   658.2% return
    USDT rsv:  $     514 (2.3% of pf)

  PORTFOLIO:   $ 22,155.19
  P&L:         $ 19,005.19
  RETURN:        603.34%   (ann 24.97%)
  MAX DD:         56.56%
  SHARPE:          1.08
  PROFIT FACT:     3.52
  CALMAR:          0.44
  WIN RATE:       64.4%  (67/104 months)
  AVG WIN:       19.57%   AVG LOSS:  -10.07%
  BEST MONTH:   238.75%   WORST:     -30.79%

  SELLS:       11 events
    2017-12: BNB SELL 35% @ 9
    2018-04: BNB SELL 35% @ 14
    2019-04: BNB SELL 35% @ 22
    2020-11: BTC SELL 35% @ 19696
    2020-12: BNB SELL 35% @ 37
    2021-01: BTC SELL 35% @ 33093
    2021-02: BNB SELL 35% @ 210
    2021-03: BTC SELL 35% @ 58741
    2021-04: BNB SELL 35% @ 623

In [7]:
# --- CHART ---
fig=plt.figure(figsize=(18, 16))
gs=fig.add_gridspec(6, 3, height_ratios=[3, 1.2, 1.2, 1.2, 0.8, 0.8])

ax=fig.add_subplot(gs[0, :])
ax.fill_between(res['date'], res['inj'], res['pf'],
    where=res['pf']>=res['inj'], color='#22c55e', alpha=0.08)
ax.fill_between(res['date'], res['pf'], res['inj'],
    where=res['pf']<res['inj'], color='#ef4444', alpha=0.08)
ax.plot(res['date'], res['inj'], color='#6b7280', lw=1.2, ls='--', label='Cost Basis')
ax.plot(res['date'], res['pf'], color='#2563eb', lw=2, label='Portfolio')
ax.scatter(sells['date'], sells['pf'], color='#dc2626', s=60, marker='v',
    zorder=5, edgecolors='white', linewidth=0.6, label=f'Sell {TP_PERCENTAGE*100:.0f}% ({len(sells)})')
ret_pct=(pf_val/inj-1)*100
sign='+' if ret_pct>=0 else ''
ax.text(res['date'].iloc[-1], pf_val, f'  ${pf_val:,.0f}  ({sign}{ret_pct:.0f}%)',
    va='bottom', fontsize=11, fontweight='bold', color='#2563eb')
ax.set_ylabel('USDT', fontsize=10)
ax.set_title(f'DCA Backtest Lab - {len(PAIRS)} pairs, sell {TP_PERCENTAGE*100:.0f}% at {TP_MULTIPLIER}x ATH',
    fontsize=13, fontweight='bold', pad=10)
ax.legend(loc='upper left', fontsize=9, ncol=3, framealpha=0.9)
ax.grid(True, alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax=fig.add_subplot(gs[1, :])
ax.fill_between(res['date'], 0, ddr*100, color='#ef4444', alpha=0.25)
ax.plot(res['date'], ddr*100, color='#dc2626', lw=0.8)
ax.axhline(y=maxdd, color='#991b1b', ls=':', lw=1.2, label=f'Max DD: {maxdd:.1f}%')
ax.set_ylabel('DD %', fontsize=10); ax.set_ylim(bottom=0, top=maxdd*1.25 if maxdd>0 else 10)
ax.legend(fontsize=9, loc='lower left'); ax.grid(True, alpha=0.15)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Allocation stack
ax=fig.add_subplot(gs[2, :])
stack_vals=[]; stack_labels=[]; stack_colors=[]
palette=['#f59e0b','#8b5cf6','#06b6d4','#ec4899','#14b8a6']
for idx,(sym,_,_,_,_) in enumerate(PAIRS):
    stack_vals.append(res[f'{sym[:3]}_val'])
    stack_labels.append(sym[:3])
    stack_colors.append(palette[idx%len(palette)])
stack_vals.append(res['usdt'])
stack_labels.append('USDT')
stack_colors.append('#10b981')
ax.stackplot(res['date'], stack_vals, labels=stack_labels, colors=stack_colors, alpha=0.8)
ax.set_ylabel('Allocation $', fontsize=10); ax.legend(fontsize=8, loc='upper left', ncol=4)
ax.grid(True, alpha=0.15); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Individual asset charts
for idx,(sym,_,_,_,_) in enumerate(PAIRS):
    ax=fig.add_subplot(gs[3, idx])
    col=palette[idx%len(palette)]
    ax.fill_between(res['date'], 0, res[f'{sym[:3]}_val'], color=col, alpha=0.12)
    ax.plot(res['date'], res[f'{sym[:3]}_val'], color=col, lw=1.5)
    ax_t=ax.twinx()
    ax_t.step(res['date'], res[f'{sym[:3]}_reinvest'], where='post', color=col, lw=0.8, alpha=0.5, ls='--')
    cap_val=state[sym]['cap']
    ax_t.set_ylabel('Reinvest %', color=col, fontsize=9); ax_t.set_ylim(0, cap_val*1.15)
    ax.set_ylabel('Value $', fontsize=9)
    ax.set_title(sym, fontsize=10, fontweight='bold'); ax.grid(True, alpha=0.15)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Reserve chart
ax=fig.add_subplot(gs[3, len(PAIRS)]) if len(PAIRS)<3 else fig.add_subplot(gs[4, 0])
if len(PAIRS)<3:
    ax.fill_between(res['date'], 0, res['usdt'], color='#10b981', alpha=0.15)
    ax.plot(res['date'], res['usdt'], color='#059669', lw=1.5)
    ax.scatter(sells['date'], sells['usdt'], color='#dc2626', s=30, marker='o', zorder=4, alpha=0.6)
    ax.set_ylabel('USDT Reserve', fontsize=9)
    ax.set_title('Reserve (spikes=sells, dips=reinvest)', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.15); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Price charts
for idx,(sym,_,_,_,_) in enumerate(PAIRS):
    ax=fig.add_subplot(gs[4 if len(PAIRS)<3 else 5, idx])
    col=palette[idx%len(palette)]
    ax.plot(res['date'], data[sym]['c'], color=col, lw=1.5)
    ax.fill_between(res['date'], 0, data[sym]['c'], color=col, alpha=0.06)
    ax.set_ylabel('Price $', fontsize=9); ax.set_title(f'{sym} Close', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.15); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

# Return histogram
if len(PAIRS)<3:
    ax=fig.add_subplot(gs[5, :2])
    colors=['#22c55e' if v>=0 else '#ef4444' for v in mret]
    ax.bar(res['date'].iloc[1:], mret*100, color=colors, width=25, alpha=0.7)
    ax.axhline(y=0, color='#374151', lw=0.6)
    ax.set_ylabel('Monthly P&L %', fontsize=9)
    ax.set_title('Monthly Returns', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.15, axis='y'); ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

    ax=fig.add_subplot(gs[5, 2])
    ax.axis('off')
    metrics=[
        ('Portfolio', f'${pf_val:,.0f}'),
        ('Return', f'{(pf_val/inj-1)*100:.1f}%'),
        ('Ann. Return', f'{ann*100:.1f}%'),
        ('Max DD', f'{maxdd:.1f}%'),
        ('Sharpe', f'{sh:.2f}'),
        ('Calmar', f'{cal:.2f}'),
        ('PF', f'{pfr:.2f}'),
        ('Win Rate', f'{wr:.1f}%'),
        ('Sells', f'{len(sells)}'),
        ('Dip Buys', f'{len(dip_mo)} mo'),
        ('Idle', f'{f["usdt"]/pf_val*100:.1f}%'),
        ('Duration', f'{len(res)} mo'),
    ]
    tbl=ax.table(cellText=metrics, colWidths=[0.35, 0.25],
        cellLoc='left', loc='center', bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False); tbl.set_fontsize(9)

plt.tight_layout()
plt.savefig('backtest-chart.png', dpi=150, bbox_inches='tight')
print('Chart saved: backtest-chart.png')
plt.show()

Chart saved: backtest-chart.png
